In [1]:
"""
Swin Transformer fine-tuning pipeline (PyTorch + timm) pour Jupyter Notebook
Avec timer pour mesurer le temps de création des DataLoader.
"""

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
import timm
from tqdm.notebook import tqdm
from collections import Counter
from sklearn.metrics import top_k_accuracy_score

# ------------------------------ Options ------------------------------
options = {
    'data_dir': '../data/cars_dataset',  # dossier racine ImageFolder avec train/ et val/
    'num_classes': 1702,
    'model_name': 'swin_large_patch4_window7_224',
    'pretrained': True,
    'image_size': 224,
    'batch_size': 16,
    'epochs': 10,
    'head_epochs': 3,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'num_workers': 16,  # bon compromis pour CPU 24c/32t
    'use_focal': False,
    'fp16': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# ------------------------------ Dataset helpers ------------------------------
def create_transforms(image_size=224, train=True):
    if train:
        t = transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.2, hue=0.02),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    else:
        t = transforms.Compose([
            transforms.Resize(int(image_size * 256/224)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    return t

train_tf = create_transforms(options['image_size'], train=True)
val_tf = create_transforms(options['image_size'], train=False)

# ------------------------------ Chargement datasets + timer ------------------------------
start_time = time.time()

train_ds = ImageFolder(os.path.join(options['data_dir'], 'train'), transform=train_tf)
val_ds = ImageFolder(os.path.join(options['data_dir'], 'val'), transform=val_tf)

# WeightedRandomSampler optimisé
labels = [y for _, y in train_ds.samples]
counts = Counter(labels)
class_weights = {cls: len(labels) / (len(counts) * count) for cls, count in counts.items()}
weights = [class_weights[l] for l in labels]

sampler = WeightedRandomSampler(weights,
                                num_samples=len(weights),
                                replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=options['batch_size'],
    sampler=sampler,
    num_workers=options['num_workers'],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_loader = DataLoader(
    val_ds,
    batch_size=options['batch_size'],
    shuffle=False,
    num_workers=options['num_workers'],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

print(f"✅ DataLoaders prêts en {time.time() - start_time:.2f} secondes.")

# ------------------------------ Model ------------------------------
model = timm.create_model(options['model_name'],
                          pretrained=options['pretrained'],
                          num_classes=options['num_classes'])
model.to(options['device'])

# ------------------------------ Loss ------------------------------
criterion = nn.CrossEntropyLoss().to(options['device'])

# ------------------------------ Optimizer ------------------------------
optimizer = optim.AdamW(model.parameters(),
                        lr=options['lr'],
                        weight_decay=options['weight_decay'])

# ------------------------------ Training functions ------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_targets, all_preds = [], []
    for images, targets in tqdm(loader, desc='Train', leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        all_targets.extend(targets.cpu().numpy())
        all_preds.extend(outputs.detach().cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    top1 = top_k_accuracy_score(all_targets, all_preds, k=1)
    top5 = top_k_accuracy_score(all_targets, all_preds, k=5)
    return epoch_loss, top1, top5

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_targets, all_preds = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Val', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(outputs.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    top1 = top_k_accuracy_score(all_targets, all_preds, k=1)
    top5 = top_k_accuracy_score(all_targets, all_preds, k=5)
    return epoch_loss, top1, top5

# ------------------------------ Training loop ------------------------------
for epoch in range(options['epochs']):
    train_loss, train_top1, train_top5 = train_one_epoch(
        model, train_loader, criterion, optimizer, options['device']
    )
    val_loss, val_top1, val_top5 = validate(
        model, val_loader, criterion, options['device']
    )
    print(f"Epoch {epoch+1}/{options['epochs']}: "
          f"train_loss={train_loss:.4f} top1={train_top1:.4f} top5={train_top5:.4f} | "
          f"val_loss={val_loss:.4f} val_top1={val_top1:.4f} val_top5={val_top5:.4f}")


✅ DataLoaders prêts en 6.69 secondes.


model.safetensors:   0%|          | 0.00/788M [00:00<?, ?B/s]

Train:   0%|          | 0/8503 [00:00<?, ?it/s]

Val:   0%|          | 0/8503 [00:00<?, ?it/s]

Epoch 1/10: train_loss=2.2976 top1=0.6059 top5=0.7291 | val_loss=0.7205 val_top1=0.8228 val_top5=0.9529


...

Val:   0%|          | 0/8503 [00:00<?, ?it/s]

Epoch 9/10: train_loss=0.0930 top1=0.9753 top5=0.9983 | val_loss=0.1980 val_top1=0.9444 val_top5=0.9937


Train:   0%|          | 0/8503 [00:00<?, ?it/s]

Val:   0%|          | 0/8503 [00:00<?, ?it/s]

Epoch 10/10: train_loss=0.0858 top1=0.9769 top5=0.9986 | val_loss=0.2164 val_top1=0.9406 val_top5=0.9920


In [7]:
# ------------------ Sauvegarde du modèle + noms de classes ------------------
import torch

# Récupérer le mapping des classes depuis ImageFolder
class_to_idx = train_ds.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Créer un checkpoint avec les poids et le mapping
checkpoint = {
    "model_state": model.state_dict(),
    "class_to_idx": class_to_idx,
}

torch.save(checkpoint, "swin_car_model_with_classes.pth")
print("✅ Modèle et mapping des classes sauvegardés dans swin_car_model_with_classes.pth")


✅ Modèle et mapping des classes sauvegardés dans swin_car_model_with_classes.pth


In [3]:
# ------------------ Chargement du modèle et test sur une image ------------------
import torch
from torchvision import transforms
from PIL import Image
import timm

# Options à adapter
model_name = 'swin_large_patch4_window7_224'
weights_path = 'swin_car_model_with_classes.pth'  # checkpoint avec mapping
test_image_path = 'car2.jpg'

# Charger le checkpoint (poids + mapping)
checkpoint = torch.load(weights_path, map_location='cpu')

# Recréer le modèle avec le bon nombre de classes
num_classes = len(checkpoint["class_to_idx"])
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
model.load_state_dict(checkpoint["model_state"])
model.eval()

# Récupérer le mapping idx -> nom de classe
class_to_idx = checkpoint["class_to_idx"]
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Prétraitements identiques à ceux utilisés à l’entraînement
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Charger et transformer l’image
img = Image.open(test_image_path).convert('RGB')
input_tensor = transform(img).unsqueeze(0)  # ajout dimension batch

# Inférence
with torch.no_grad():
    outputs = model(input_tensor)
    probs = torch.nn.functional.softmax(outputs, dim=1)
    top5_prob, top5_idx = torch.topk(probs, 5)

print("Top-5 prédictions :")
for i in range(5):
    idx = top5_idx[0][i].item()
    prob = top5_prob[0][i].item()
    class_name = idx_to_class[idx]
    print(f"{class_name} (classe {idx}) avec probabilité {prob:.4f}")


Top-5 prédictions :
Seat LEON (classe 1332) avec probabilité 0.9925
Golf (classe 672) avec probabilité 0.0015
Golf R (classe 674) avec probabilité 0.0013
Golf GTI (classe 673) avec probabilité 0.0012
Benz E Class AMG (classe 229) avec probabilité 0.0005
